# Configuração do Ambiente

In [0]:
########################################################
#### LIBS
########################################################
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
#from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Leitura dos dados

In [0]:
### função de leitura dos dados
def leitura_prata(dados):
     return spark.read.format("parquet").load(f"/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/{dados}")

In [0]:
### leitura dos dados
alunos = leitura_prata("alunos")
municipio = leitura_prata("municipio")
uf = leitura_prata("uf")
meta_municipio = leitura_prata("meta_alfabetizacao_municipio")
meta_uf = leitura_prata("meta_alfabetizacao_uf")
meta_brasil = leitura_prata("meta_alfabetizacao_brasil")

ibge = leitura_prata("ibge_ideb")


# Construção da camada Ouro

## Município

Note que, os municípios se repetem. Dessa forma, a análise será feita **apenas** com as redes municipais de ensino.

In [0]:
###
dados_final = (
    municipio
    .filter(F.col("ano") == "2023")
    .groupBy("id_municipio")
    .count()
    .withColumnRenamed("count", "n")
    .orderBy(F.col("n").desc())
    .limit(5)
)

dados_final.show()

In [0]:
municipio2 = (municipio
    .filter(F.col("ano") == 2023)
    .filter(F.col("rede") == "3")
    .select("id_municipio", "rede", "ano", "taxa_alfabetizacao", "media_portugues")
)

municipio2.show(5)
municipio2.count()

In [0]:
municipio2.filter(F.col("id_municipio").isNull()).show()

## Meta municipal

Note que a `taxa_alfabetização` presente na tabela das metas municipais é referente a 2024. No entanto, para a construção da camada ouro, e consequentemente para o ajuste dos, serão usados apenas os dados referentes ao anos de 2023.

In [0]:
###
meta_municipio2 = (meta_municipio
    .filter(F.col("ano") == 2023)
    .select("id_municipio", "ano", "rede", "meta_alfabetizacao_2024", "meta_alfabetizacao_2030"))

meta_municipio2.show(5)
meta_municipio2.count()

In [0]:
meta_municipio2.filter(F.col("id_municipio").isNull()).show()

## Dados de fontes externas

In [0]:
###
ibge2 = (ibge
    .filter(F.col("rede") == "Municipal")
    .withColumnsRenamed({
    "CD_MUN": "id_municipio",
    "AREA_KM2": "area_km2",
    "POPULACAO": "pop"})  
)
ibge2.show(5)
print(f"(Número de Estados únicos, Número de municípios): {(ibge2.select("sigla_uf").distinct().count(), ibge2.count())}")

In [0]:
###
print(f"(Número ufs antes, número de ufs depois): {(ibge.select("sigla_uf").distinct().count(),ibge2.select("sigla_uf").distinct().count())}.")
print("O Distrito Federal é desconsiderado, uam vez que não tem rede municipal de ensino.")

## Join das tabelas

A tabela `municipio2` é a base (contém a variável `taxa_alfabetizacao`), por isso é usada em um *left join* com as demais:
- `meta_municipio2`: junção por `id_municipio` e `rede` (ambas já filtradas para 2023);
- `ibge2`: junção por `id_municipio` (a coluna `rede` do IBGE/IDEB é removida, pois contém apenas o valor "Municipal").

A chave `id_municipio` é convertida para string nas três tabelas, pois no IBGE ela foi lida como texto.

In [0]:
### join das três tabelas
dados_ouro = (municipio2
    .withColumn("id_municipio", F.col("id_municipio").cast("string"))
    .join(
        meta_municipio2
            .withColumn("id_municipio", F.col("id_municipio").cast("string"))
            .drop("ano"),
        on="id_municipio", how="left")
    .join(
        ibge2
            .withColumn("id_municipio", F.col("id_municipio").cast("string"))
            .drop("rede"),
        on="id_municipio", how="left")
    .drop("ano", "rede")
)

dados_ouro.show(5)
dados_ouro.count()

In [0]:
### verificação do join: duplicidade de municípios
print("linhas:", dados_ouro.count())
print("municípios distintos:", dados_ouro.select("id_municipio").distinct().count())

In [0]:
dados_ouro.filter(F.col("sigla_uf").isNull()).show()

## Imputação dos dados

Os valores ausentes das colunas `meta_alfabetizacao_2024` (meta municipal) e `matematica_saeb`, `portugues_saeb` e `ideb_2023` (IBGE/IDEB) são imputados pela **mediana da UF**.

A UF é obtida pelos dois primeiros dígitos do código IBGE do município (`cd_uf`), pois a coluna `sigla_uf` vem do IDEB e fica nula para os municípios sem correspondência no join.

In [0]:
### colunas a serem imputadas
colunas_imputar = ["meta_alfabetizacao_2024", "matematica_saeb", "portugues_saeb", "ideb_2023"]

### código da UF (2 primeiros dígitos do código IBGE do município)
#dados_ouro = dados_ouro.withColumn("cd_uf", F.substring(F.col("id_municipio"), 1, 2))

### quantidade de valores ausentes antes da imputação
dados_ouro.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_imputar]).show()

In [0]:
### mediana de cada coluna por UF
medianas_uf = (dados_ouro
    .groupBy("sigla_uf")
    .agg(*[F.median(c).alias(f"mediana_{c}") for c in colunas_imputar])
)

medianas_uf.orderBy("sigla_uf").show(27)

In [0]:
### imputação: valores nulos recebem a mediana da UF
dados_ouro = (dados_ouro
    .join(medianas_uf, on="cd_uf", how="left")
    .withColumns({c: F.coalesce(F.col(c), F.col(f"mediana_{c}")) for c in colunas_imputar})
    .drop(*[f"mediana_{c}" for c in colunas_imputar])
)

### quantidade de valores ausentes após a imputação
dados_ouro.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_imputar]).show()

dados_ouro.show(5)
dados_ouro.count()

#Enriquecimento da base de dados - criação de novas features/colunas

# Exportação dos dados

# Rascunhos - Exemplos

In [0]:
dados_ouro.filter(F.col("id_municipio") == "5300108")

In [0]:
"""
##
dados_final = (
    municipio2
    .groupBy("id_municipio")
    .count()
    .orderBy(F.col("count"))
    .withColumnRenamed("count", "n")
    .show()
)
##
x = municipio.select("id_municipio").distinct()
x.show()

###
remover_colunas = [c for c in alunos.columns if c.startswith("_")]

alunos2 = (alunos
    .filter(F.col("presenca") == 1)
    .drop(*remover_colunas)
    .drop("peso_aluno", "presenca", "preenchimento_caderno", "serie"))
"""
#(municipio2.count(), len(municipio2.columns))

#municipio2.filter(~F.col("id_municipio").isin(x)).show()

#dados_filtrado = dados.filter(~F.col("x").isin(v))

#meta_municipio.select("rede").distinct().show()

#dados_filtrado = dados.filter(F.col("a") == "3w")

